# 🔬 Perbandingan Model: Faster R-CNN vs YOLOv8m
## Deteksi Pelanggaran di Laboratorium

Notebook ini membandingkan performa dua model object detection:
- **Faster R-CNN** (ResNet50 FPN v2) - `best_model-gabungan6.pth`
- **YOLOv8m** - `best.pt`

Data test diambil dari **Roboflow**, model diload dari **Google Drive**.

---
### 📋 Kelas yang Dideteksi:
| ID | Kelas | Deskripsi |
|---|---|---|
| 1 | `playing game phone` | Main HP / Game |
| 2 | `sleep` | Tidur |
| 3 | `eating` | Makan |

---
## 📦 BAGIAN 1: Instalasi Library

In [ ]:
# ==========================================
# CELL 1: INSTALASI LIBRARY
# ==========================================
# Install semua library yang dibutuhkan
!pip install roboflow torch torchvision torchmetrics ultralytics -q
!pip install albumentations seaborn pandas numpy matplotlib opencv-python-headless -q

print("✅ Instalasi selesai!")

---
## 📚 BAGIAN 2: Import Library & Konfigurasi

In [ ]:
# ==========================================
# CELL 2: IMPORT LIBRARY
# ==========================================
import os
import glob
import zipfile
import shutil
import time
import json
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

import torch
import torch.utils.data
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights
from torchmetrics.detection.mean_ap import MeanAveragePrecision

from ultralytics import YOLO
from roboflow import Roboflow
from IPython.display import display, HTML

# Device
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"🖥️ Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ==========================================
# CELL 3: KONFIGURASI UTAMA
# ==========================================

# ============================================================
# ⚙️ SESUAIKAN KONFIGURASI INI SEBELUM MENJALANKAN NOTEBOOK
# ============================================================

# --- Konfigurasi Google Drive ---
# Path ke model Faster R-CNN di Google Drive
FRCNN_MODEL_DRIVE_PATH = "/content/drive/MyDrive/best_model-gabungan6.pth"

# Path ke model YOLOv8m di Google Drive
YOLO_MODEL_DRIVE_PATH = "/content/drive/MyDrive/best.pt"

# --- Konfigurasi Roboflow ---
ROBOFLOW_API_KEY = "qkrwwn0zGtWk0FFezlwt"  # Ganti dengan API Key Anda
WORKSPACE_NAME = "datasetbariq"
PROJECT_NAME = "violations-in-lab"
VERSION_NUMBER = 1

# --- Konfigurasi Evaluasi ---
CONF_THRESHOLD = 0.5    # Threshold confidence untuk deteksi
IOU_THRESHOLD  = 0.5    # Threshold IoU untuk matching TP

# --- Mapping Kelas ---
CLASS_MAP = {
    'background': 0,
    'playing game phone': 1,
    'sleep': 2,
    'eating': 3
}
NUM_CLASSES = 4  # termasuk background

# Display name untuk laporan
DISPLAY_NAMES = {
    'playing game phone': 'Game (Main HP)',
    'sleep': 'Tidur',
    'eating': 'Makan',
}

# Folder dataset
DATASET_FOLDER = "/content/dataset_test"

print("✅ Konfigurasi berhasil dimuat!")
print(f"   Faster R-CNN model: {FRCNN_MODEL_DRIVE_PATH}")
print(f"   YOLOv8m model    : {YOLO_MODEL_DRIVE_PATH}")
print(f"   Conf Threshold   : {CONF_THRESHOLD}")
print(f"   IoU Threshold    : {IOU_THRESHOLD}")

---
## ☁️ BAGIAN 3: Mount Google Drive & Load Model

In [ ]:
# ==========================================
# CELL 4: MOUNT GOOGLE DRIVE
# ==========================================
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive berhasil di-mount!")

In [ ]:
# ==========================================
# CELL 5: LOAD KEDUA MODEL
# ==========================================

# --- 1. Load Faster R-CNN ---
print("🔄 Loading Faster R-CNN (ResNet50 FPN v2)...")
assert os.path.exists(FRCNN_MODEL_DRIVE_PATH), \
    f"❌ File tidak ditemukan: {FRCNN_MODEL_DRIVE_PATH}\nPastikan model sudah diupload ke Google Drive!"

# Bangun arsitektur Faster R-CNN
frcnn_model = fasterrcnn_resnet50_fpn_v2(weights=None)
in_features = frcnn_model.roi_heads.box_predictor.cls_score.in_features
frcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# Load bobot yang sudah dilatih
checkpoint = torch.load(FRCNN_MODEL_DRIVE_PATH, map_location=DEVICE)
frcnn_model.load_state_dict(checkpoint)
frcnn_model.to(DEVICE)
frcnn_model.eval()
print("✅ Faster R-CNN berhasil diload!")

print()

# --- 2. Load YOLOv8m ---
print("🔄 Loading YOLOv8m...")
assert os.path.exists(YOLO_MODEL_DRIVE_PATH), \
    f"❌ File tidak ditemukan: {YOLO_MODEL_DRIVE_PATH}\nPastikan model sudah diupload ke Google Drive!"

yolo_model = YOLO(YOLO_MODEL_DRIVE_PATH)
print("✅ YOLOv8m berhasil diload!")

print()
print("🎉 Kedua model siap digunakan!")

---
## 📥 BAGIAN 4: Download Data Test dari Roboflow

In [ ]:
# ==========================================
# CELL 6: DOWNLOAD DATA TEST DARI ROBOFLOW
# ==========================================
# Download format VOC (Pascal VOC / XML) untuk Faster R-CNN
# Format ini berisi folder train/valid/test dengan file .xml annotation

if os.path.exists(DATASET_FOLDER):
    shutil.rmtree(DATASET_FOLDER)
    print("🗑️ Folder dataset lama dihapus.")

print("⏳ Mendownload data dari Roboflow...")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
dataset = project.version(VERSION_NUMBER).download(
    "voc",
    location=DATASET_FOLDER
)

print(f"✅ Dataset berhasil didownload ke: {DATASET_FOLDER}")

# Cek struktur folder
print("\n📁 Struktur folder dataset:")
for root, dirs, files in os.walk(DATASET_FOLDER):
    level = root.replace(DATASET_FOLDER, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        img_count = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        xml_count = len([f for f in files if f.endswith('.xml')])
        if img_count > 0:
            print(f"{subindent}📸 {img_count} gambar, 📄 {xml_count} annotation XML")

In [ ]:
# ==========================================
# CELL 7: SIAPKAN DATA TEST
# ==========================================

# Tentukan folder test
test_dir = os.path.join(DATASET_FOLDER, "test")

if not os.path.exists(test_dir):
    # Jika tidak ada folder test, gunakan valid sebagai pengganti
    test_dir = os.path.join(DATASET_FOLDER, "valid")
    print(f"⚠️ Folder 'test' tidak ditemukan, menggunakan folder 'valid': {test_dir}")
else:
    print(f"✅ Menggunakan folder test: {test_dir}")

# Hitung jumlah file
all_imgs = sorted([f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
all_xmls = sorted([f for f in os.listdir(test_dir) if f.endswith('.xml')])

print(f"\n📊 Statistik Data Test:")
print(f"   Total gambar      : {len(all_imgs)}")
print(f"   Total annotation  : {len(all_xmls)}")

# Hitung distribusi kelas dari annotation
class_counts = {cls: 0 for cls in CLASS_MAP if cls != 'background'}
for xml_file in all_xmls:
    xml_path = os.path.join(test_dir, xml_file)
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for obj in root.findall('object'):
            label = obj.find('name').text.strip().lower()
            # Normalisasi
            label = label.replace('-', ' ').replace('_', ' ')
            if label in class_counts:
                class_counts[label] += 1
    except:
        pass

print(f"\n📋 Distribusi Ground Truth per Kelas:")
for cls, count in class_counts.items():
    display_name = DISPLAY_NAMES.get(cls, cls)
    print(f"   {display_name:<20}: {count} bounding box")

---
## 🔧 BAGIAN 5: Fungsi Helper untuk Evaluasi

In [ ]:
# ==========================================
# CELL 8: DATASET CLASS (untuk Faster R-CNN)
# ==========================================

class LabViolationDataset(torch.utils.data.Dataset):
    """Dataset class untuk format Pascal VOC (XML annotation)"""

    def __init__(self, root, class_map=None):
        self.root = root
        self.class_map = class_map or CLASS_MAP

        all_files = list(sorted(os.listdir(root)))
        self.imgs = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, self.imgs[idx])
        img_filename = self.imgs[idx]
        xml_filename = os.path.splitext(img_filename)[0] + '.xml'
        annot_path = os.path.join(self.root, xml_filename)

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            return self.__getitem__((idx + 1) % len(self.imgs))

        try:
            tree = ET.parse(annot_path)
        except ET.ParseError:
            return self.__getitem__((idx + 1) % len(self.imgs))

        root_xml = tree.getroot()
        boxes = []
        labels = []

        for member in root_xml.findall('object'):
            label_text = member.find('name').text.strip().lower()
            label_text = label_text.replace('-', ' ').replace('_', ' ')

            if label_text not in self.class_map:
                continue

            labels.append(self.class_map[label_text])
            bndbox = member.find('bndbox')
            xmin = int(float(bndbox.find('xmin').text))
            ymin = int(float(bndbox.find('ymin').text))
            xmax = int(float(bndbox.find('xmax').text))
            ymax = int(float(bndbox.find('ymax').text))
            boxes.append([xmin, ymin, xmax, ymax])

        if len(boxes) == 0:
            return self.__getitem__((idx + 1) % len(self.imgs))

        boxes  = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(boxes),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        # Convert image to tensor [0, 1]
        img_tensor = torchvision.transforms.ToTensor()(img)

        return img_tensor, target, img_path

    def __len__(self):
        return len(self.imgs)


def collate_fn(batch):
    return tuple(zip(*batch))


# Buat dataset & loader untuk Faster R-CNN
test_dataset = LabViolationDataset(test_dir)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=2
)

print(f"✅ Dataset siap! Jumlah gambar: {len(test_dataset)}")

In [ ]:
# ==========================================
# CELL 9: FUNGSI EVALUASI
# ==========================================

def box_iou_calc(box1, box2):
    """Hitung IoU matrix antara dua set bounding box."""
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    lt = torch.max(box1[:, None, :2], box2[:, :2])
    rb = torch.min(box1[:, None, 2:], box2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2 - inter
    return inter / (union + 1e-6)


def evaluate_frcnn(model, data_loader, device,
                   conf_threshold=CONF_THRESHOLD, iou_threshold=IOU_THRESHOLD):
    """
    Evaluasi model Faster R-CNN.
    Menghitung mAP, Precision, Recall, F1-Score, dan FPS.
    """
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

    tp_total, fp_total, fn_total = 0, 0, 0
    total_gt = 0
    inference_times = []

    # Counter per kelas
    rev_class_map = {v: k for k, v in CLASS_MAP.items() if v != 0}
    class_ids = sorted(rev_class_map.keys())
    class_tp = {cid: 0 for cid in class_ids}
    class_gt = {cid: 0 for cid in class_ids}

    with torch.no_grad():
        for images, targets, _ in data_loader:
            images_gpu = [img.to(device) for img in images]
            targets_gpu = [{k: v.to(device) for k, v in t.items()} for t in targets]

            # Ukur waktu inferensi
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_start = time.time()
            outputs = model(images_gpu)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_end = time.time()
            inference_times.append((t_end - t_start) / len(images))

            # Update mAP metric
            metric.update(outputs, targets_gpu)

            # Hitung TP/FP/FN manual
            for i, output in enumerate(outputs):
                pred_boxes  = output['boxes']
                pred_scores = output['scores']
                pred_labels = output['labels']
                gt_boxes    = targets_gpu[i]['boxes']
                gt_labels   = targets_gpu[i]['labels']

                # Filter per kelas GT
                for cid in class_ids:
                    class_gt[cid] += (gt_labels == cid).sum().item()

                total_gt += len(gt_boxes)

                # Filter confidence
                mask = pred_scores >= conf_threshold
                pred_boxes  = pred_boxes[mask]
                pred_labels = pred_labels[mask]

                if len(pred_boxes) == 0:
                    fn_total += len(gt_boxes)
                    continue
                if len(gt_boxes) == 0:
                    fp_total += len(pred_boxes)
                    continue

                iou_matrix = box_iou_calc(pred_boxes, gt_boxes)
                matched_gt = set()

                for p_idx in range(len(pred_boxes)):
                    best_iou, best_gt_idx = iou_matrix[p_idx].max(dim=0)
                    if (best_iou >= iou_threshold and
                            pred_labels[p_idx] == gt_labels[best_gt_idx] and
                            best_gt_idx.item() not in matched_gt):
                        tp_total += 1
                        matched_gt.add(best_gt_idx.item())
                        cid = pred_labels[p_idx].item()
                        if cid in class_tp:
                            class_tp[cid] += 1
                    else:
                        fp_total += 1

    fn_total = total_gt - tp_total

    # Hitung metrik global
    precision = tp_total / (tp_total + fp_total + 1e-6)
    recall    = tp_total / (tp_total + fn_total + 1e-6)
    f1        = 2 * precision * recall / (precision + recall + 1e-6)

    # Hitung mAP
    map_results = metric.compute()
    map50     = map_results['map_50'].item()
    map50_95  = map_results['map'].item()
    map_per_class = map_results.get('map_per_class', None)

    avg_inf_ms = np.mean(inference_times) * 1000
    fps = 1000.0 / avg_inf_ms if avg_inf_ms > 0 else 0

    # Per-kelas metrics
    per_class_rows = []
    for i, cid in enumerate(class_ids):
        name = rev_class_map[cid]
        gt_c  = class_gt[cid]
        tp_c  = class_tp[cid]
        fn_c  = gt_c - tp_c
        rec_c = (tp_c / gt_c * 100) if gt_c > 0 else 0.0
        map_c = map_per_class[i].item() if (map_per_class is not None and i < len(map_per_class)) else 0.0
        per_class_rows.append({
            'Kelas': DISPLAY_NAMES.get(name, name),
            'GT (BB)': gt_c,
            'TP': tp_c,
            'FN': fn_c,
            'Recall (%)': f"{rec_c:.1f}%",
            'mAP50-95': f"{map_c:.3f}",
        })

    return {
        'model_name': 'Faster R-CNN',
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'mAP@0.5': map50,
        'mAP@0.5:0.95': map50_95,
        'avg_inference_ms': avg_inf_ms,
        'fps': fps,
        'tp': tp_total,
        'fp': fp_total,
        'fn': fn_total,
        'total_gt': total_gt,
        'per_class': pd.DataFrame(per_class_rows)
    }


def evaluate_yolo(yolo_model, test_dir, class_map,
                  conf_threshold=CONF_THRESHOLD, iou_threshold=IOU_THRESHOLD):
    """
    Evaluasi model YOLOv8.
    Menggunakan built-in metrics dari Ultralytics YOLO.
    """
    rev_class_map = {v: k for k, v in class_map.items() if v != 0}
    class_ids = sorted(rev_class_map.keys())

    # Parse semua gambar dan annotation dari test_dir
    img_files = sorted([f for f in os.listdir(test_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

    tp_total, fp_total, fn_total = 0, 0, 0
    total_gt = 0
    inference_times = []

    class_tp = {cid: 0 for cid in class_ids}
    class_gt = {cid: 0 for cid in class_ids}

    # Simpan prediksi dan GT untuk mAP (format torchmetrics)
    all_preds = []
    all_targets = []

    # Mapping YOLO class index -> CLASS_MAP index
    # YOLOv8 biasanya menggunakan class index 0-based, sesuaikan dengan class_map kita
    # Kita perlu tahu nama kelas yang dipelajari model YOLO
    yolo_class_names = yolo_model.names  # dict {0: 'eating', 1: 'playing game phone', 2: 'sleep'} misalnya
    print(f"   YOLO class names: {yolo_class_names}")

    # Buat mapping dari YOLO class index ke CLASS_MAP index
    yolo_to_classmap = {}
    for yolo_idx, yolo_name in yolo_class_names.items():
        yolo_name_clean = yolo_name.strip().lower().replace('-', ' ').replace('_', ' ')
        if yolo_name_clean in class_map:
            yolo_to_classmap[yolo_idx] = class_map[yolo_name_clean]
    print(f"   YOLO -> CLASS_MAP mapping: {yolo_to_classmap}")

    for img_file in img_files:
        img_path = os.path.join(test_dir, img_file)
        xml_path = os.path.join(test_dir, os.path.splitext(img_file)[0] + '.xml')

        if not os.path.exists(xml_path):
            continue

        # Parse GT
        try:
            tree = ET.parse(xml_path)
            root_xml = tree.getroot()
        except:
            continue

        gt_boxes_list = []
        gt_labels_list = []
        for member in root_xml.findall('object'):
            label_text = member.find('name').text.strip().lower()
            label_text = label_text.replace('-', ' ').replace('_', ' ')
            if label_text not in class_map or class_map[label_text] == 0:
                continue
            bndbox = member.find('bndbox')
            xmin = int(float(bndbox.find('xmin').text))
            ymin = int(float(bndbox.find('ymin').text))
            xmax = int(float(bndbox.find('xmax').text))
            ymax = int(float(bndbox.find('ymax').text))
            gt_boxes_list.append([xmin, ymin, xmax, ymax])
            gt_labels_list.append(class_map[label_text])

        if len(gt_boxes_list) == 0:
            continue

        gt_boxes  = torch.tensor(gt_boxes_list, dtype=torch.float32)
        gt_labels = torch.tensor(gt_labels_list, dtype=torch.int64)

        for cid in class_ids:
            class_gt[cid] += (gt_labels == cid).sum().item()
        total_gt += len(gt_boxes)

        # Inferensi YOLO
        t_start = time.time()
        results = yolo_model.predict(
            img_path,
            conf=conf_threshold,
            iou=iou_threshold,
            verbose=False
        )
        t_end = time.time()
        inference_times.append(t_end - t_start)

        # Parse hasil YOLO
        result = results[0]
        if result.boxes is not None and len(result.boxes) > 0:
            pred_boxes_np  = result.boxes.xyxy.cpu().numpy()
            pred_scores_np = result.boxes.conf.cpu().numpy()
            pred_cls_np    = result.boxes.cls.cpu().numpy().astype(int)

            # Convert YOLO class idx -> CLASS_MAP idx
            pred_labels_mapped = []
            valid_mask = []
            for ci in pred_cls_np:
                if ci in yolo_to_classmap:
                    pred_labels_mapped.append(yolo_to_classmap[ci])
                    valid_mask.append(True)
                else:
                    valid_mask.append(False)

            valid_mask = np.array(valid_mask)
            if valid_mask.sum() > 0:
                pred_boxes_t  = torch.tensor(pred_boxes_np[valid_mask], dtype=torch.float32)
                pred_scores_t = torch.tensor(pred_scores_np[valid_mask], dtype=torch.float32)
                pred_labels_t = torch.tensor(pred_labels_mapped, dtype=torch.int64)
            else:
                pred_boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
                pred_scores_t = torch.zeros(0, dtype=torch.float32)
                pred_labels_t = torch.zeros(0, dtype=torch.int64)
        else:
            pred_boxes_t  = torch.zeros((0, 4), dtype=torch.float32)
            pred_scores_t = torch.zeros(0, dtype=torch.float32)
            pred_labels_t = torch.zeros(0, dtype=torch.int64)

        # Simpan untuk mAP calculation
        all_preds.append({
            'boxes': pred_boxes_t,
            'scores': pred_scores_t,
            'labels': pred_labels_t
        })
        all_targets.append({
            'boxes': gt_boxes,
            'labels': gt_labels
        })

        # Hitung TP/FP/FN manual
        if len(pred_boxes_t) == 0:
            fn_total += len(gt_boxes)
        elif len(gt_boxes) == 0:
            fp_total += len(pred_boxes_t)
        else:
            iou_matrix = box_iou_calc(pred_boxes_t, gt_boxes)
            matched_gt = set()
            for p_idx in range(len(pred_boxes_t)):
                best_iou, best_gt_idx = iou_matrix[p_idx].max(dim=0)
                if (best_iou >= iou_threshold and
                        pred_labels_t[p_idx] == gt_labels[best_gt_idx] and
                        best_gt_idx.item() not in matched_gt):
                    tp_total += 1
                    matched_gt.add(best_gt_idx.item())
                    cid = pred_labels_t[p_idx].item()
                    if cid in class_tp:
                        class_tp[cid] += 1
                else:
                    fp_total += 1

    fn_total = total_gt - tp_total

    # Hitung metrik global
    precision = tp_total / (tp_total + fp_total + 1e-6)
    recall    = tp_total / (tp_total + fn_total + 1e-6)
    f1        = 2 * precision * recall / (precision + recall + 1e-6)

    # Hitung mAP dengan torchmetrics
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
    metric.update(all_preds, all_targets)
    map_results   = metric.compute()
    map50         = map_results['map_50'].item()
    map50_95      = map_results['map'].item()
    map_per_class = map_results.get('map_per_class', None)

    avg_inf_ms = np.mean(inference_times) * 1000
    fps = 1000.0 / avg_inf_ms if avg_inf_ms > 0 else 0

    # Per-kelas metrics
    per_class_rows = []
    for i, cid in enumerate(class_ids):
        name  = rev_class_map[cid]
        gt_c  = class_gt[cid]
        tp_c  = class_tp[cid]
        fn_c  = gt_c - tp_c
        rec_c = (tp_c / gt_c * 100) if gt_c > 0 else 0.0
        map_c = map_per_class[i].item() if (map_per_class is not None and i < len(map_per_class)) else 0.0
        per_class_rows.append({
            'Kelas': DISPLAY_NAMES.get(name, name),
            'GT (BB)': gt_c,
            'TP': tp_c,
            'FN': fn_c,
            'Recall (%)': f"{rec_c:.1f}%",
            'mAP50-95': f"{map_c:.3f}",
        })

    return {
        'model_name': 'YOLOv8m',
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'mAP@0.5': map50,
        'mAP@0.5:0.95': map50_95,
        'avg_inference_ms': avg_inf_ms,
        'fps': fps,
        'tp': tp_total,
        'fp': fp_total,
        'fn': fn_total,
        'total_gt': total_gt,
        'per_class': pd.DataFrame(per_class_rows)
    }


print("✅ Fungsi evaluasi berhasil dimuat!")

---
## 🏃 BAGIAN 6: Jalankan Evaluasi Kedua Model

In [ ]:
# ==========================================
# CELL 10: EVALUASI FASTER R-CNN
# ==========================================
print("=" * 60)
print("🔍 Mengevaluasi Faster R-CNN...")
print("=" * 60)

frcnn_results = evaluate_frcnn(
    frcnn_model,
    test_loader,
    DEVICE,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD
)

print("\n✅ Evaluasi Faster R-CNN selesai!")
print(f"   mAP@0.5         : {frcnn_results['mAP@0.5']:.4f}")
print(f"   mAP@0.5:0.95    : {frcnn_results['mAP@0.5:0.95']:.4f}")
print(f"   Precision       : {frcnn_results['precision']:.4f}")
print(f"   Recall          : {frcnn_results['recall']:.4f}")
print(f"   F1-Score        : {frcnn_results['f1_score']:.4f}")
print(f"   Avg Inference   : {frcnn_results['avg_inference_ms']:.1f} ms/gambar")
print(f"   FPS             : {frcnn_results['fps']:.1f}")

In [ ]:
# ==========================================
# CELL 11: EVALUASI YOLOv8m
# ==========================================
print("=" * 60)
print("🔍 Mengevaluasi YOLOv8m...")
print("=" * 60)

yolo_results = evaluate_yolo(
    yolo_model,
    test_dir,
    CLASS_MAP,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD
)

print("\n✅ Evaluasi YOLOv8m selesai!")
print(f"   mAP@0.5         : {yolo_results['mAP@0.5']:.4f}")
print(f"   mAP@0.5:0.95    : {yolo_results['mAP@0.5:0.95']:.4f}")
print(f"   Precision       : {yolo_results['precision']:.4f}")
print(f"   Recall          : {yolo_results['recall']:.4f}")
print(f"   F1-Score        : {yolo_results['f1_score']:.4f}")
print(f"   Avg Inference   : {yolo_results['avg_inference_ms']:.1f} ms/gambar")
print(f"   FPS             : {yolo_results['fps']:.1f}")

---
## 📊 BAGIAN 7: Tabel Perbandingan

In [ ]:
# ==========================================
# CELL 12: TABEL PERBANDINGAN UTAMA
# ==========================================

print("\n" + "=" * 70)
print("📊 TABEL PERBANDINGAN HASIL EVALUASI")
print("=" * 70)

comparison_data = {
    'Metrik': [
        'mAP@0.5',
        'mAP@0.5:0.95',
        'Precision',
        'Recall',
        'F1-Score',
        'Avg Inference (ms)',
        'FPS',
        'TP', 'FP', 'FN',
        'Total GT (BB)'
    ],
    'Faster R-CNN': [
        f"{frcnn_results['mAP@0.5']:.4f}",
        f"{frcnn_results['mAP@0.5:0.95']:.4f}",
        f"{frcnn_results['precision']:.4f}",
        f"{frcnn_results['recall']:.4f}",
        f"{frcnn_results['f1_score']:.4f}",
        f"{frcnn_results['avg_inference_ms']:.1f} ms",
        f"{frcnn_results['fps']:.1f}",
        str(frcnn_results['tp']),
        str(frcnn_results['fp']),
        str(frcnn_results['fn']),
        str(frcnn_results['total_gt'])
    ],
    'YOLOv8m': [
        f"{yolo_results['mAP@0.5']:.4f}",
        f"{yolo_results['mAP@0.5:0.95']:.4f}",
        f"{yolo_results['precision']:.4f}",
        f"{yolo_results['recall']:.4f}",
        f"{yolo_results['f1_score']:.4f}",
        f"{yolo_results['avg_inference_ms']:.1f} ms",
        f"{yolo_results['fps']:.1f}",
        str(yolo_results['tp']),
        str(yolo_results['fp']),
        str(yolo_results['fn']),
        str(yolo_results['total_gt'])
    ]
}

df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.set_index('Metrik')

# Tampilkan tabel
display(df_comparison.style.set_properties(**{
    'text-align': 'center',
    'font-size': '14px'
}).set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4CAF50'), ('color', 'white'),
                                  ('font-size', '14px'), ('text-align', 'center')]},
    {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f2f2f2')]}
]))

# Tentukan pemenang
print("\n📌 Ringkasan Perbandingan:")
metrics_to_compare = [
    ('mAP@0.5', frcnn_results['mAP@0.5'], yolo_results['mAP@0.5']),
    ('mAP@0.5:0.95', frcnn_results['mAP@0.5:0.95'], yolo_results['mAP@0.5:0.95']),
    ('Precision', frcnn_results['precision'], yolo_results['precision']),
    ('Recall', frcnn_results['recall'], yolo_results['recall']),
    ('F1-Score', frcnn_results['f1_score'], yolo_results['f1_score']),
    ('FPS (lebih tinggi = lebih baik)', frcnn_results['fps'], yolo_results['fps']),
]

for metric_name, frcnn_val, yolo_val in metrics_to_compare:
    if frcnn_val > yolo_val:
        winner = "✅ Faster R-CNN unggul"
    elif yolo_val > frcnn_val:
        winner = "✅ YOLOv8m unggul"
    else:
        winner = "🤝 Seri"
    print(f"   {metric_name:<35}: {winner}")

In [ ]:
# ==========================================
# CELL 13: TABEL PERBANDINGAN PER KELAS
# ==========================================

print("\n📋 HASIL PER KELAS - Faster R-CNN:")
print("-" * 60)
display(frcnn_results['per_class'])

print("\n📋 HASIL PER KELAS - YOLOv8m:")
print("-" * 60)
display(yolo_results['per_class'])

---
## 📈 BAGIAN 8: Visualisasi Perbandingan

In [ ]:
# ==========================================
# CELL 14: VISUALISASI GRAFIK PERBANDINGAN
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Perbandingan Model: Faster R-CNN vs YOLOv8m', fontsize=16, fontweight='bold', y=1.02)

models = ['Faster R-CNN', 'YOLOv8m']
colors = ['#2196F3', '#FF9800']

# --- Plot 1: Metrik Akurasi ---
ax1 = axes[0]
metrics_names = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'F1-Score']
frcnn_vals = [
    frcnn_results['mAP@0.5'],
    frcnn_results['mAP@0.5:0.95'],
    frcnn_results['precision'],
    frcnn_results['recall'],
    frcnn_results['f1_score']
]
yolo_vals = [
    yolo_results['mAP@0.5'],
    yolo_results['mAP@0.5:0.95'],
    yolo_results['precision'],
    yolo_results['recall'],
    yolo_results['f1_score']
]

x = np.arange(len(metrics_names))
w = 0.35
bars1 = ax1.bar(x - w/2, frcnn_vals, w, label='Faster R-CNN', color=colors[0], alpha=0.85)
bars2 = ax1.bar(x + w/2, yolo_vals, w, label='YOLOv8m', color=colors[1], alpha=0.85)
ax1.set_xlabel('Metrik')
ax1.set_ylabel('Nilai')
ax1.set_title('Metrik Akurasi', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_names, rotation=25, ha='right', fontsize=9)
ax1.set_ylim(0, 1.1)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

# --- Plot 2: Kecepatan Inferensi ---
ax2 = axes[1]
fps_vals = [frcnn_results['fps'], yolo_results['fps']]
inf_ms_vals = [frcnn_results['avg_inference_ms'], yolo_results['avg_inference_ms']]

bar_colors = [colors[0], colors[1]]
bars_fps = ax2.bar(models, fps_vals, color=bar_colors, alpha=0.85)
ax2.set_xlabel('Model')
ax2.set_ylabel('FPS (Frame Per Detik)')
ax2.set_title('Kecepatan Inferensi (FPS)', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for bar, ms in zip(bars_fps, inf_ms_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{bar.get_height():.1f} FPS\n({ms:.1f} ms)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# --- Plot 3: Recall per Kelas (Grouped Bar) ---
ax3 = axes[2]
class_names_plot = [row['Kelas'] for _, row in frcnn_results['per_class'].iterrows()]
frcnn_recall_pct = [
    float(row['Recall (%)'].replace('%', '')) / 100
    for _, row in frcnn_results['per_class'].iterrows()
]
yolo_recall_pct = [
    float(row['Recall (%)'].replace('%', '')) / 100
    for _, row in yolo_results['per_class'].iterrows()
]

x3 = np.arange(len(class_names_plot))
bars3a = ax3.bar(x3 - w/2, frcnn_recall_pct, w, label='Faster R-CNN', color=colors[0], alpha=0.85)
bars3b = ax3.bar(x3 + w/2, yolo_recall_pct, w, label='YOLOv8m', color=colors[1], alpha=0.85)
ax3.set_xlabel('Kelas')
ax3.set_ylabel('Recall')
ax3.set_title('Recall per Kelas', fontweight='bold')
ax3.set_xticks(x3)
ax3.set_xticklabels(class_names_plot, fontsize=10)
ax3.set_ylim(0, 1.15)
ax3.legend()
ax3.grid(axis='y', alpha=0.3)
for bar in bars3a:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars3b:
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('/content/comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik perbandingan disimpan ke /content/comparison_chart.png")

In [ ]:
# ==========================================
# CELL 15: RADAR CHART PERBANDINGAN
# ==========================================

categories = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'F1-Score']
N = len(categories)

frcnn_values = [
    frcnn_results['mAP@0.5'],
    frcnn_results['mAP@0.5:0.95'],
    frcnn_results['precision'],
    frcnn_results['recall'],
    frcnn_results['f1_score']
]
yolo_values = [
    yolo_results['mAP@0.5'],
    yolo_results['mAP@0.5:0.95'],
    yolo_results['precision'],
    yolo_results['recall'],
    yolo_results['f1_score']
]

# Tutup polygon
frcnn_values += frcnn_values[:1]
yolo_values  += yolo_values[:1]

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)

ax.plot(angles, frcnn_values, 'o-', linewidth=2, color='#2196F3', label='Faster R-CNN')
ax.fill(angles, frcnn_values, alpha=0.25, color='#2196F3')

ax.plot(angles, yolo_values, 'o-', linewidth=2, color='#FF9800', label='YOLOv8m')
ax.fill(angles, yolo_values, alpha=0.25, color='#FF9800')

ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.7)

plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15), fontsize=12)
plt.title('Radar Chart Perbandingan Model', size=14, fontweight='bold', pad=20)
plt.savefig('/content/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Radar chart disimpan ke /content/radar_chart.png")

---
## 🖼️ BAGIAN 9: Visualisasi Deteksi pada Gambar Sample

In [ ]:
# ==========================================
# CELL 16: VISUALISASI DETEKSI SIDE-BY-SIDE
# ==========================================
# Tampilkan beberapa gambar sample dengan deteksi dari kedua model

NUM_SAMPLES = 5    # Jumlah gambar yang ditampilkan
CONF_VIZ    = CONF_THRESHOLD  # Threshold untuk visualisasi

CLASS_COLORS = {
    1: (255,  85,  85),   # playing game phone -> merah
    2: ( 85,  85, 255),   # sleep              -> biru
    3: ( 85, 200,  85),   # eating             -> hijau
}
GT_COLOR = (200, 200, 0)  # Ground truth -> kuning

rev_class_map = {v: k for k, v in CLASS_MAP.items() if v != 0}


def draw_boxes_on_img(img_np, boxes, labels, scores, color_map, thickness=2, show_score=True):
    """Gambar bounding box di atas gambar."""
    img_out = img_np.copy()
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        label_id = int(labels[i])
        color = color_map.get(label_id, (128, 128, 128))
        cv2.rectangle(img_out, (x1, y1), (x2, y2), color, thickness)
        name = rev_class_map.get(label_id, str(label_id))
        display_name = DISPLAY_NAMES.get(name, name)
        if show_score and scores is not None:
            text = f"{display_name}: {float(scores[i]):.2f}"
        else:
            text = display_name
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img_out, (x1, y1 - th - 5), (x1 + tw + 4, y1), color, -1)
        cv2.putText(img_out, text, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return img_out


# Pilih sample gambar secara acak
sample_imgs = sorted(os.listdir(test_dir))
sample_imgs = [f for f in sample_imgs if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
sample_indices = np.random.choice(len(sample_imgs), min(NUM_SAMPLES, len(sample_imgs)), replace=False)

frcnn_model.eval()

for idx in sample_indices:
    img_file = sample_imgs[idx]
    img_path = os.path.join(test_dir, img_file)
    xml_path = os.path.join(test_dir, os.path.splitext(img_file)[0] + '.xml')

    # Load gambar
    img_pil = Image.open(img_path).convert("RGB")
    img_np  = np.array(img_pil)

    # --- Faster R-CNN inference ---
    img_tensor = torchvision.transforms.ToTensor()(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        frcnn_out = frcnn_model(img_tensor)[0]
    frcnn_boxes  = frcnn_out['boxes'].cpu().numpy()
    frcnn_scores = frcnn_out['scores'].cpu().numpy()
    frcnn_labels = frcnn_out['labels'].cpu().numpy()
    mask_frcnn = frcnn_scores >= CONF_VIZ
    frcnn_boxes  = frcnn_boxes[mask_frcnn]
    frcnn_scores = frcnn_scores[mask_frcnn]
    frcnn_labels = frcnn_labels[mask_frcnn]

    # --- YOLOv8m inference ---
    yolo_out = yolo_model.predict(img_path, conf=CONF_VIZ, verbose=False)[0]
    if yolo_out.boxes is not None and len(yolo_out.boxes) > 0:
        yolo_boxes_raw  = yolo_out.boxes.xyxy.cpu().numpy()
        yolo_scores_raw = yolo_out.boxes.conf.cpu().numpy()
        yolo_cls_raw    = yolo_out.boxes.cls.cpu().numpy().astype(int)
        # Map YOLO class -> CLASS_MAP
        yolo_names = yolo_model.names
        yolo_to_cm = {}
        for yi, yn in yolo_names.items():
            yn_clean = yn.strip().lower().replace('-', ' ').replace('_', ' ')
            if yn_clean in CLASS_MAP:
                yolo_to_cm[yi] = CLASS_MAP[yn_clean]
        yolo_mapped = [yolo_to_cm.get(c, -1) for c in yolo_cls_raw]
        valid = [i for i, m in enumerate(yolo_mapped) if m > 0]
        yolo_boxes  = yolo_boxes_raw[valid]
        yolo_scores = yolo_scores_raw[valid]
        yolo_labels = np.array([yolo_mapped[i] for i in valid])
    else:
        yolo_boxes  = np.zeros((0, 4))
        yolo_scores = np.zeros(0)
        yolo_labels = np.zeros(0, dtype=int)

    # --- Ground Truth ---
    gt_boxes_list, gt_labels_list = [], []
    if os.path.exists(xml_path):
        try:
            tree = ET.parse(xml_path)
            root_xml = tree.getroot()
            for member in root_xml.findall('object'):
                lbl = member.find('name').text.strip().lower().replace('-', ' ').replace('_', ' ')
                if lbl in CLASS_MAP and CLASS_MAP[lbl] != 0:
                    bndbox = member.find('bndbox')
                    gt_boxes_list.append([
                        int(float(bndbox.find('xmin').text)),
                        int(float(bndbox.find('ymin').text)),
                        int(float(bndbox.find('xmax').text)),
                        int(float(bndbox.find('ymax').text))
                    ])
                    gt_labels_list.append(CLASS_MAP[lbl])
        except:
            pass

    # --- Gambar hasil ---
    img_gt    = draw_boxes_on_img(img_np, gt_boxes_list, gt_labels_list, None,
                                  {k: GT_COLOR for k in CLASS_COLORS}, show_score=False)
    img_frcnn = draw_boxes_on_img(img_np, frcnn_boxes, frcnn_labels, frcnn_scores, CLASS_COLORS)
    img_yolo  = draw_boxes_on_img(img_np, yolo_boxes, yolo_labels, yolo_scores, CLASS_COLORS)

    # Plot side-by-side
    fig, axs = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'Gambar: {img_file}', fontsize=13, fontweight='bold')

    axs[0].imshow(cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB) if len(img_gt.shape)==3 else img_gt)
    axs[0].set_title(f'Ground Truth ({len(gt_boxes_list)} obj)', fontweight='bold')
    axs[0].axis('off')

    axs[1].imshow(cv2.cvtColor(img_frcnn, cv2.COLOR_BGR2RGB) if len(img_frcnn.shape)==3 else img_frcnn)
    axs[1].set_title(f'Faster R-CNN ({len(frcnn_boxes)} deteksi)', fontweight='bold')
    axs[1].axis('off')

    axs[2].imshow(cv2.cvtColor(img_yolo, cv2.COLOR_BGR2RGB) if len(img_yolo.shape)==3 else img_yolo)
    axs[2].set_title(f'YOLOv8m ({len(yolo_boxes)} deteksi)', fontweight='bold')
    axs[2].axis('off')

    # Legend warna
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=tuple(c/255 for c in GT_COLOR), label='Ground Truth (kuning)'),
        Patch(facecolor=tuple(c/255 for c in CLASS_COLORS[1]), label='Game/Main HP'),
        Patch(facecolor=tuple(c/255 for c in CLASS_COLORS[2]), label='Tidur'),
        Patch(facecolor=tuple(c/255 for c in CLASS_COLORS[3]), label='Makan'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=10,
               bbox_to_anchor=(0.5, -0.05))

    plt.tight_layout()
    plt.show()
    print("-" * 80)

---
## 💾 BAGIAN 10: Simpan Hasil ke Google Drive

In [ ]:
# ==========================================
# CELL 17: SIMPAN HASIL KE GOOGLE DRIVE
# ==========================================

SAVE_DIR = "/content/drive/MyDrive/hasil_perbandingan_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Simpan tabel perbandingan ke CSV
df_comparison.to_csv(f"{SAVE_DIR}/comparison_table.csv")
frcnn_results['per_class'].to_csv(f"{SAVE_DIR}/frcnn_per_class.csv", index=False)
yolo_results['per_class'].to_csv(f"{SAVE_DIR}/yolo_per_class.csv", index=False)

# Simpan grafik
shutil.copy('/content/comparison_chart.png', f"{SAVE_DIR}/comparison_chart.png")
shutil.copy('/content/radar_chart.png', f"{SAVE_DIR}/radar_chart.png")

# Simpan hasil lengkap ke JSON
results_json = {
    'faster_rcnn': {
        'mAP@0.5': frcnn_results['mAP@0.5'],
        'mAP@0.5:0.95': frcnn_results['mAP@0.5:0.95'],
        'precision': frcnn_results['precision'],
        'recall': frcnn_results['recall'],
        'f1_score': frcnn_results['f1_score'],
        'avg_inference_ms': frcnn_results['avg_inference_ms'],
        'fps': frcnn_results['fps'],
        'tp': frcnn_results['tp'],
        'fp': frcnn_results['fp'],
        'fn': frcnn_results['fn'],
        'total_gt': frcnn_results['total_gt'],
    },
    'yolov8m': {
        'mAP@0.5': yolo_results['mAP@0.5'],
        'mAP@0.5:0.95': yolo_results['mAP@0.5:0.95'],
        'precision': yolo_results['precision'],
        'recall': yolo_results['recall'],
        'f1_score': yolo_results['f1_score'],
        'avg_inference_ms': yolo_results['avg_inference_ms'],
        'fps': yolo_results['fps'],
        'tp': yolo_results['tp'],
        'fp': yolo_results['fp'],
        'fn': yolo_results['fn'],
        'total_gt': yolo_results['total_gt'],
    },
    'config': {
        'conf_threshold': CONF_THRESHOLD,
        'iou_threshold': IOU_THRESHOLD,
        'test_dir': test_dir,
    }
}

with open(f"{SAVE_DIR}/evaluation_results.json", 'w') as f:
    json.dump(results_json, f, indent=2)

print(f"✅ Hasil evaluasi berhasil disimpan ke: {SAVE_DIR}")
print(f"   📄 comparison_table.csv")
print(f"   📄 frcnn_per_class.csv")
print(f"   📄 yolo_per_class.csv")
print(f"   🖼️ comparison_chart.png")
print(f"   🖼️ radar_chart.png")
print(f"   📋 evaluation_results.json")

---
## 🎯 Kesimpulan

Notebook ini telah melakukan evaluasi komprehensif terhadap dua model deteksi objek:

| Aspek | Faster R-CNN | YOLOv8m |
|---|---|---|
| **Arsitektur** | Two-stage detector | One-stage detector |
| **Keunggulan** | Akurasi tinggi, presisi detail | Kecepatan inferensi tinggi |
| **Cocok untuk** | Sistem dengan GPU dedicated | Real-time / edge computing |

### Metrik yang Dievaluasi:
- **mAP@0.5**: Mean Average Precision pada IoU threshold 0.5
- **mAP@0.5:0.95**: Mean Average Precision pada range IoU 0.5-0.95 (standar COCO)
- **Precision**: Dari semua deteksi, berapa persen yang benar
- **Recall**: Dari semua objek real, berapa persen yang berhasil dideteksi
- **F1-Score**: Harmonic mean dari Precision dan Recall
- **FPS**: Frame Per Second (kecepatan inferensi)

> 💡 **Catatan**: Conf Threshold dan IoU Threshold dapat disesuaikan di Cell 3 (KONFIGURASI UTAMA) untuk analisis lebih lanjut.